# 04 · Demand model v3: factorised network, negative binomial, calibrated intervals

Trains and scores the v3 model under the fixed protocol of `docs/evaluation.md` (`citibike_eval.py`).
Models and predictions are cached in `tmp/v3/`, so a rerun takes a few minutes. Delete the folder to retrain everything (about 90 minutes on 8 CPU cores).

| Part | What it answers |
| --- | --- |
| 1 | Which variant to freeze, decided on the validation block |
| 2 | How it does on the five rolling folds |
| 3 | The shared-shock variances that calibrate the intervals, fitted on the folds only |
| 4 | Whether a level correction helps (a frozen model predicting five weeks ahead) |
| 5 | The frozen candidate on the test block, scored once |
| 6 | Interval coverage on the test block |
| 7 | ONNX export for the dashboard |

**Design.** For station s, slot t and target c: `log mu = b[s,c] + g0[t,c] + sum_r U[s,r,c] G[t,r,c]`.
A network over the calendar and weather of the slot produces `g0` and `G` once per slot, and each station supplies `b` and `U` from a free embedding, a neighbourhood embedding and its coordinates.
Training cost per row is a dot product, so a fit takes about 6 minutes instead of 20 minutes per 14 epochs for the v2 recipe on 8 times less data.
The likelihood is a negative binomial whose dispersion depends on the rate, the target and the station.

In [1]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

import citibike_eval as ce
import citibike_intervals as ci
import citibike_model as cm
import citibike_v2recipe as v2r

REPO = Path('..')
DATA_DIR = Path('data_procesada')
RESULTS = REPO / 'data' / 'results'
CACHE = REPO / 'tmp' / 'cache'
WORK = REPO / 'tmp' / 'v3'
MODELS = Path('modelos')
for p in (RESULTS, WORK, MODELS):
  p.mkdir(parents=True, exist_ok=True)
torch.set_num_threads(max(1, __import__('os').cpu_count()))
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)
quiet = lambda *a, **k: None

def save(df, name):
  df.round(6).to_csv(RESULTS / name, index=False)
  print(f'saved {RESULTS / name} ({len(df):,} rows)')

## 1. Data and variants
The rate network sees 13 inputs that the ONNX contract can provide (calendar, temperature, humidity, wind, rain, rain of the previous 2 hours, weather code).

In [2]:
data = ce.load_processed(DATA_DIR, cache_dir=CACHE)
ds = cm.Dataset(data)
st = cm.station_inputs(data, REPO)
d_train_end, d_val_end = data.day(ce.TRAIN_END) + 1, data.day(ce.VAL[1]) + 1
val, test = data.block('val', *ce.VAL), data.block('test', *ce.TEST)
groups = ce.station_groups(data, 0, d_train_end, REPO)
region, borough = np.asarray(groups['nta']), np.asarray(groups['borough'])
print(f'{data.n_est:,} stations | {data.n_days} days | {st[2]} neighbourhood groups | train ends day {d_train_end}')

2,515 stations | 365 days | 122 neighbourhood groups | train ends day 337


In [3]:
VARIANTS = {'nb_prior': dict(), 'poisson': dict(dist='poisson'), 'nb_noprior': dict(use_prior=False), 'nb_rank48': dict(rank=48)}
rows = []
for name, over in VARIANTS.items():
  model, feats = cm.fit_cached(data, ds, st, 0, d_train_end, dict(epochs=16, **over), WORK / f'{name}_parity.pt', log=quiet)
  mu, al = cm.predict(model, feats, val)
  rows.append(ce.block_summary(data, name, mu, al if over.get('dist', 'nb') == 'nb' else None, val, 'train-only'))
ablation = pd.DataFrame(rows)
save(ablation, 'v3_ablation_val.csv')
ablation[['model', 'deviance_15min', 'wape_hour', 'wape_day', 'bias', 'logscore_poisson', 'logscore_nb']].round(4)

saved ../data/results/v3_ablation_val.csv (4 rows)


,model,deviance_15min,wape_hour,wape_day,bias,logscore_poisson,logscore_nb
0,nb_prior,0.8699,0.4496,0.1908,0.9468,0.5567,0.5462
1,poisson,0.8693,0.4498,0.1910,0.9486,0.5565,NaN
2,nb_noprior,0.8745,0.4529,0.1945,0.9493,0.5578,0.5472
3,nb_rank48,0.8694,0.4488,0.1913,0.9432,0.5565,0.5461


Reading the table (validation block, every variant fitted on days before Sep 3):
- Poisson and negative binomial give the same point forecast. The negative binomial is chosen for the distribution: its log score is lower.
- The neighbourhood embedding and coordinates help.
- Rank 48 gains nothing over rank 24.

The frozen candidate is `nb_prior`: negative binomial, neighbourhood prior, rank 24, 16 epochs.

## 2. Rolling origin
Five weekly folds, each fitted on everything before it, scored against the baseline chosen on validation.

In [4]:
pm_base = pd.read_csv(RESULTS / 'point_metrics.csv')
fold_pred, rows = {}, []
for name, first, last in ce.FOLDS:
  fold = data.block(name, first, last)
  f = WORK / f'fold_{name}.npz'
  if f.exists():
    z = np.load(f)
    mu, al = z['mu'], z['al']
  else:
    model, feats = cm.fit_cached(data, ds, st, 0, fold.d0, dict(epochs=16), WORK / f'fold_{name}.pt', log=quiet)
    mu, al = cm.predict(model, feats, fold)
    np.savez_compressed(f, mu=mu, al=al)
  fold_pred[name] = (fold, mu, al)
  rows.append(ce.block_summary(data, 'v3 (NegBin)', mu, al, fold, 'rolling'))
folds = pd.DataFrame(rows)
bar = ce.summary(pm_base[pm_base['fit'] == 'rolling']).loc['C|all+level']
mean = folds[['deviance_15min', 'mae_15min', 'wape_hour', 'wape_day', 'bias']].mean()
print('Mean over folds, v3 against the baseline to beat (C|all+level):')
print(pd.DataFrame({'v3': mean, 'bar': bar[mean.index], 'gain %': (1 - mean / bar[mean.index]) * 100}).round(4))
save(folds, 'v3_folds.csv')
folds.round(4)

Mean over folds, v3 against the baseline to beat (C|all+level):
                    v3     bar   gain %
deviance_15min  0.8653  0.9363   7.5888
mae_15min       0.5666  0.5763   1.6710
wape_hour       0.4701  0.4948   4.9808
wape_day        0.1904  0.2338  18.5750
bias            0.9942  0.9913  -0.2873
saved ../data/results/v3_folds.csv (5 rows)


,model,fit,block,deviance_15min,mae_15min,wape_hour,wape_day,bias,logscore_poisson,logscore_nb
0,v3 (NegBin),rolling,fold1,0.8780,0.5740,0.4599,0.1729,0.9944,0.5530,0.5405
1,v3 (NegBin),rolling,fold2,0.8548,0.5581,0.4727,0.1830,1.0288,0.5343,0.5244
2,v3 (NegBin),rolling,fold3,0.8583,0.5513,0.5190,0.2167,1.0467,0.5201,0.5094
3,v3 (NegBin),rolling,fold4,0.8663,0.5699,0.4683,0.1882,1.0220,0.5462,0.5356
4,v3 (NegBin),rolling,fold5,0.8691,0.5800,0.4306,0.1912,0.8791,0.5659,0.5559


## 3. Shared shocks and calibrated intervals

A cell's total over-dispersion `alpha` mixes independent noise with shocks that many cells share: the weather of a day or an hour, a neighbourhood event, a station out of service.
Summing independent cell intervals ignores that and is far too narrow for station-days (coverage 0.53 for the Poisson assumption, see `docs/evaluation.md`).
Five shock variances per bike type are fitted by non-negative least squares on the excess variance of six aggregation families (station, neighbourhood and city, by hour and by day), using only the five held-out folds.
See `citibike_intervals.py` for the formulas.

In [5]:
blocks_cal = []
for name, first, last in ce.FOLDS:
  fold, mu, al = fold_pred[name]
  Yb, ab, _ = ce.block_arrays(data, fold)
  blocks_cal.append((np.asarray(Yb), mu, al, np.asarray(ab)))
shocks = ci.fit_shocks(blocks_cal, region)
pd.DataFrame({b: {k: s[k] for k in ci.SHOCKS} for b, s in shocks.items()}).round(4)

,classic,electric
D,0.0031,0.0046
H,0.0248,0.0197
R,0.0132,0.0070
Ds,0.0496,0.0385
Hs,0.0937,0.0413


In [6]:
def plain(x):
  return {k: plain(v) for k, v in x.items()} if isinstance(x, dict) else float(x)

(RESULTS / 'v3_shocks.json').write_text(json.dumps(plain(shocks), indent=1))
pd.DataFrame({b: s['ratios'] for b, s in shocks.items()}).round(4)

,classic,electric
station-hour,0.1843,0.1111
station-day,0.0738,0.0541
region-hour,0.0478,0.0350
region-day,0.0198,0.0144
city-hour,0.0280,0.0242
city-day,0.0114,0.0101


## 4. Does a level correction help?

The brief proposed a recent-demand scaling term.
Test it with a model frozen at Aug 12 that predicts the next five weeks: the factor for each week is the previous week's actual over predicted ratio (clipped to 0.8 to 1.25).

In [7]:
f = WORK / 'frozen_drift.csv'
if f.exists():
  drift = pd.read_csv(f)
else:
  f1 = data.block('fold1', *ce.FOLDS[0][1:])
  span = data.block('span', ce.FOLDS[0][1], ce.FOLDS[-1][2])
  model, feats = cm.fit_cached(data, ds, st, 0, f1.d0, dict(epochs=16), WORK / 'fold_fold1.pt', log=quiet)
  mu5, al5 = cm.predict(model, feats, span)
  weeks = [data.block(f'w{i + 1}', a, b) for i, (n, a, b) in enumerate(ce.FOLDS)]
  out, prev = [], None
  for i, w in enumerate(weeks):
    sl = slice((w.d0 - span.d0) * 96, (w.d1 - span.d0) * 96)
    Yw, aw, _ = ce.block_arrays(data, w)
    Yw, aw = np.asarray(Yw), np.asarray(aw)
    mu = mu5[:, sl]
    ratio = (Yw * aw[..., None]).sum((0, 1)) / (mu * aw[..., None]).sum((0, 1))
    row = dict(week=w.name, horizon_weeks=i + 1, ratio=np.round(ratio, 3).tolist(),
               **{k: v for k, v in ce.block_summary(data, 'frozen', mu, None, w).items() if k in ('deviance_15min', 'wape_hour', 'wape_day', 'bias')})
    if prev is not None:
      c = ce.block_summary(data, 'corrected', mu * np.clip(prev, 0.8, 1.25)[None, None, :], None, w)
      row.update(deviance_15min_corrected=c['deviance_15min'], wape_day_corrected=c['wape_day'], bias_corrected=c['bias'])
    out.append(row)
    prev = ratio
  drift = pd.DataFrame(out)
  drift.to_csv(f, index=False)
save(drift, 'v3_frozen_drift.csv')
drift.round(4)

saved ../data/results/v3_frozen_drift.csv (5 rows)


,week,horizon_weeks,ratio_actual_over_pred,dev15,wape_hour,wape_day,bias,dev15_corr,wape_hour_corr,wape_day_corr,bias_corr,factor
0,w1,1,"[1.0, 1.011, 1.0, 1.012]",0.8780,0.4599,0.1729,0.9944,NaN,NaN,NaN,NaN,NaN
1,w2,2,"[0.962, 0.979, 0.961, 0.979]",0.8615,0.4772,0.1878,1.0292,0.8619,0.4784,0.1891,1.0376,"[1.0, 1.011, 1.0, 1.012]"
2,w3,3,"[0.953, 0.958, 0.954, 0.959]",0.8620,0.5221,0.2215,1.0476,0.8608,0.5177,0.2190,1.0207,"[0.962, 0.979, 0.961, 0.979]"
3,w4,4,"[0.979, 0.977, 0.977, 0.976]",0.8747,0.4744,0.1964,1.0263,0.8744,0.4690,0.1940,0.9823,"[0.953, 0.958, 0.954, 0.959]"
4,w5,5,"[1.147, 1.139, 1.145, 1.138]",0.8816,0.4375,0.1993,0.8794,0.8867,0.4407,0.2110,0.8591,"[0.979, 0.977, 0.977, 0.976]"


The weekly bias moves between 0.96 and 1.14 with no persistence, so the previous week's ratio is a poor guide: the correction helps two weeks and hurts two.
**No level correction is used.**
The model also does not degrade over the five weeks it was frozen.

## 5. Frozen candidate on the test block

`train-only` is fitted on days before Sep 3 (the cut-off v2 used) and `refit` on days before Sep 17.
Reference rows come from `EVALUACION_protocolo.ipynb` (`data/results/point_metrics.csv`).
The "v2 recipe, 12 months" row is the v2 architecture, features and loss trained on the same 12-month table, with the same number of steps, to separate the effect of the data from the effect of the new architecture.

In [8]:
model_to, feats_to = cm.fit_cached(data, ds, st, 0, d_train_end, dict(epochs=16), WORK / 'nb_prior_parity.pt', log=quiet)
model_rf, feats_rf = cm.fit_cached(data, ds, st, 0, d_val_end, dict(epochs=16), WORK / 'final_refit.pt', log=quiet)
mu_to, al_to = cm.predict(model_to, feats_to, test)
mu_rf, al_rf = cm.predict(model_rf, feats_rf, test)
Yt, at, slt = (np.asarray(a) if not hasattr(a, 'iloc') else a for a in ce.block_arrays(data, test))

f = WORK / 'v2recipe_blocks.npz'
if f.exists():
  mu_v2r = np.load(f)['test']
else:
  feats_p = cm.build_features(data, d_train_end)
  m, _ = v2r.fit_v2(ds, feats_p, 0, d_train_end, log=quiet)
  mu_v2r = v2r.predict_v2(m, feats_p, test)
  np.savez_compressed(f, test=mu_v2r)

rows = [ce.block_summary(data, 'v3 (NegBin)', mu_to, al_to, test, 'train-only'),
        ce.block_summary(data, 'v3 (NegBin)', mu_rf, al_rf, test, 'refit'),
        ce.block_summary(data, 'v2 recipe, 12 months', mu_v2r, None, test, 'train-only')]
v3_test = pd.DataFrame(rows)
ref = ce.summary(pm_base[(pm_base['block'] == 'test') & pm_base['fit'].isin(['train-only', 'train-only (shipped)'])])
cols = ['deviance_15min', 'mae_15min', 'wape_hour', 'wape_day', 'bias']
table = pd.concat([v3_test.set_index(['model', 'fit'])[cols], ref.loc[['v2 network', 'C|all+level', 'C|all']].assign(fit='train-only').set_index('fit', append=True)[cols]])
print('Test block, mean of departures and arrivals totals:')
print(table.round(4))
for name in ('v2 network', 'C|all+level'):
  r = v3_test.iloc[1]
  print(f'v3 refit vs {name}:', {k: f'{(1 - r[k] / ref.loc[name, k]) * 100:+.1f}%' for k in cols[:4]})
save(v3_test, 'v3_test_summary.csv')

Test block, mean of departures and arrivals totals:
                                 deviance_15min  mae_15min  wape_hour  wape_day    bias
model                fit                                                               
v3 (NegBin)          train-only          0.7970     0.5073     0.4724    0.2161  0.9149
                     refit               0.7910     0.5060     0.4678    0.2086  0.9238
v2 recipe, 12 months train-only          0.8036     0.5098     0.4784    0.2313  0.9086
v2 network           train-only          0.8302     0.5163     0.4974    0.2555  0.8871
C|all+level          train-only          0.9111     0.5658     0.5674    0.3339  1.0693
C|all                train-only          0.9447     0.5409     0.5758    0.4077  0.7824
v3 refit vs v2 network: {'deviance_15min': '+4.7%', 'mae_15min': '+2.0%', 'wape_hour': '+5.9%', 'wape_day': '+18.3%'}
v3 refit vs C|all+level: {'deviance_15min': '+13.2%', 'mae_15min': '+10.6%', 'wape_hour': '+17.5%', 'wape_day': '+37.5%'}
save

In [9]:
# the full metric table of v3 on the test block, same format as point_metrics.csv
parts = []
for name, fit, mu in (('v3 (NegBin)', 'train-only', mu_to), ('v3 (NegBin)', 'refit', mu_rf), ('v2 recipe, 12 months', 'train-only', mu_v2r)):
  m = ce.point_metrics(Yt, mu, at)
  m.insert(0, 'model', name)
  m.insert(1, 'fit', fit)
  m.insert(2, 'block', 'test')
  parts.append(m)
for name, first, last in ce.FOLDS:
  fold, mu, al = fold_pred[name]
  Yf, af, _ = ce.block_arrays(data, fold)
  m = ce.point_metrics(Yf, mu, af)
  m.insert(0, 'model', 'v3 (NegBin)')
  m.insert(1, 'fit', 'rolling')
  m.insert(2, 'block', name)
  parts.append(m)
save(pd.concat(parts, ignore_index=True), 'point_metrics_v3.csv')

saved ../data/results/point_metrics_v3.csv (144 rows)


### Breakdowns on the test block

In [10]:
chosen = {'v3 refit': mu_rf, 'v3 train-only': mu_to}
parts = []
for name, P in chosen.items():
  for target in ['salidas_total', 'llegadas_total']:
    b = ce.breakdown(Yt, P, at, slt, groups, target=target)
    b.insert(0, 'model', name)
    parts.append(b)
breakdown = pd.concat(parts, ignore_index=True)
save(breakdown, 'breakdown_test_v3.csv')
old = pd.read_csv(RESULTS / 'breakdown_test.csv')

def compare(dimension, target='salidas_total'):
  a = pd.concat([old[old['model'].isin(['v2 network', 'C|all+level (bar)'])], breakdown[breakdown['model'] == 'v3 train-only']])
  a = a[(a['dimension'] == dimension) & (a['target'] == target)]
  return a.pivot_table(index='group', columns='model', values=['deviance', 'wape', 'bias']).round(3)

compare('volume_tier').loc[['no training data', '<2', '2-5', '5-15', '15-40', '>40']]

saved ../data/results/breakdown_test_v3.csv (660 rows)


bias                                   deviance                                       wape                         
model            C|all+level (bar) v2 network v3 train-only C|all+level (bar) v2 network v3 train-only C|all+level (bar) v2 network v3 train-only
group                                                                                                                                            
no training data             1.083      2.512         0.780             1.273      1.528         0.452             1.821      2.877         0.908
<2                           0.812      6.132         0.774             0.150      0.174         0.072             1.732      7.062         1.723
2-5                          1.058      0.906         0.853             0.400      0.283         0.281             1.887      1.761         1.718
5-15                         1.046      0.889         0.879             0.538      0.491         0.485             1.685      1.576         1.569
15-40                        1.063      0.888         0.909             0.837      0.804         0.794             1.299      1.205         1.214
>40                          1.074      0.880         0.919             1.420      1.285         1.230             0.714      0.639         0.631

In [11]:
compare('borough').drop(index='unassigned', errors='ignore')

bias                                   deviance                                       wape                         
model     C|all+level (bar) v2 network v3 train-only C|all+level (bar) v2 network v3 train-only C|all+level (bar) v2 network v3 train-only
group                                                                                                                                     
Bronx                 1.011      0.868         0.854             0.534      0.518         0.507             1.540      1.466         1.446
Brooklyn              1.096      0.928         0.933             0.878      0.788         0.746             0.939      0.866         0.842
Manhattan             1.062      0.869         0.910             1.324      1.195         1.143             0.705      0.633         0.623
Queens                1.085      0.907         0.914             0.629      0.586         0.577             1.272      1.179         1.177

In [12]:
compare('day_type')

bias                                   deviance                                       wape                         
model   C|all+level (bar) v2 network v3 train-only C|all+level (bar) v2 network v3 train-only C|all+level (bar) v2 network v3 train-only
group                                                                                                                                   
weekend              1.54      0.944         1.029             0.991      0.737         0.692             1.344      0.965         0.951
workday              0.97      0.877         0.890             0.863      0.855         0.825             0.730      0.723         0.708

## 6. Interval coverage on the test block
Nominal 80% and 90%.
Cell intervals are the per-cell negative binomial.
Aggregate intervals use the five shocks fitted on the folds.
Net flow is arrivals minus departures per station-day, and the cumulative net flow since midnight is the quantity behind the dashboard's stock band.

In [13]:
t0 = time.time()
ev = ci.evaluate(Yt, mu_rf, al_rf, at, shocks, region, groups={'borough': borough})
net = ci.evaluate_net(Yt, mu_rf, al_rf, at, shocks)
cells_cov = ci.evaluate_cells(Yt, mu_rf, al_rf, at)
print(f'{time.time() - t0:.0f}s')
save(ev, 'intervals_v3_test.csv')
save(net, 'intervals_v3_net_test.csv')
cov = ev.groupby(['group', 'window', 'level'])['coverage'].mean().unstack().round(3)
cov.index = [f'{g} {w}' for g, w in cov.index]
cov

10s
saved ../data/results/intervals_v3_test.csv (64 rows)
saved ../data/results/intervals_v3_net_test.csv (28 rows)


level,0.8,0.9
borough day,0.546,0.729
borough hour,0.760,0.850
city day,0.446,0.750
city hour,0.656,0.767
region day,0.673,0.798
region hour,0.853,0.920
station day,0.842,0.911
station hour,0.938,0.969


In [14]:
net.groupby(['quantity', 'level'])['coverage'].mean().unstack().round(3)

level,0.8,0.9
quantity,,
cumulative net to 06:00,0.914,0.949
cumulative net to 09:00,0.881,0.932
cumulative net to 12:00,0.890,0.941
cumulative net to 15:00,0.905,0.950
cumulative net to 18:00,0.919,0.957
cumulative net to 21:00,0.926,0.959
station-day net,0.929,0.961


In [15]:
ll = pd.DataFrame({'v3 NegBin': [ce.logscore_mean(Yt, at, mu_rf, al_rf)], 'v3 means with Poisson': [ce.logscore_mean(Yt, at, mu_rf)],
                   'v2 means with Poisson': [ce.logscore_mean(Yt, at, np.asarray(ce.v2_predict(data, test, REPO / 'ui/models/demanda_net_v2.onnx', REPO / 'ui/models/station_ids.json')))]})
print('Log score on the test block (lower is better):')
ll.round(4)

Log score on the test block (lower is better):


,v3 NegBin,v3 means with Poisson,v2 means with Poisson
0,0.4846,0.4925,0.5026


Station-level intervals are calibrated.
Neighbourhood, borough and city totals per day under-cover on this block: it holds the two wettest days of the year (24.5 and 49.8 mm) and several cold dry days, which is a regime the folds do not contain.
The cell-level coverage is above nominal because most counts are 0 and a discrete interval cannot hit 80% exactly.

## 7. Export for the dashboard

The ONNX keeps the v2 inputs (`slot_dia`, `dia_semana`, `es_festivo`, `clima[S, 6]`) and the first output `lambda[S, stations, 4]`.
It adds a second output `alpha[S, stations, 4]`, the negative binomial dispersion of one cell.
A JSON file holds the shock variances and the station list in model order, with an `active` flag for stations that had trips in the last 28 days.

In [16]:
import onnxruntime as ort

slots = data.slots.iloc[test.s0:test.s1]
idx = np.arange(0, 96 * 3, 11)
cols = ['temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'precipitation', 'precip_prev2h', 'weather_code']
sample = slots.iloc[idx]
example = (torch.tensor(sample['slot_dia'].values, dtype=torch.long), torch.tensor(sample['dia_semana'].values, dtype=torch.long),
           torch.tensor(sample['es_festivo'].values, dtype=torch.float32), torch.tensor(sample[cols].values, dtype=torch.float32))
onnx_path = MODELS / 'demanda_net_v3.onnx'
method = cm.export_onnx(model_rf, feats_rf.mu, feats_rf.sd, onnx_path, example)
sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
lam, alpha = sess.run(['lambda', 'alpha'], {n: t.numpy() for n, t in zip(['slot_dia', 'dia_semana', 'es_festivo', 'clima'], example)})
ref_mu = mu_rf[:, idx].transpose(1, 0, 2)
ref_al = al_rf[:, idx].transpose(1, 0, 2)
print(f'{method}: {onnx_path} ({onnx_path.stat().st_size / 1e6:.1f} MB)')
print('lambda', lam.shape, 'max abs diff vs PyTorch:', float(np.abs(lam - ref_mu).max()), '| alpha max abs diff:', float(np.abs(alpha - ref_al).max()))
assert np.abs(lam - ref_mu).max() < 1e-3 and np.abs(alpha - ref_al).max() < 1e-3

[torch.onnx] Obtain model graph for `DemandV3ONNX([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `DemandV3ONNX([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
torch.export (dynamo): modelos/demanda_net_v3.onnx (1.2 MB)
lambda (27, 2515, 4) max abs diff vs PyTorch: 7.62939453125e-06 | alpha max abs diff: 2.86102294921875e-05


W1010 09:29:17.928000 23462 site-packages/torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::nms
W1010 09:29:17.928000 23462 site-packages/torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::roi_align
W1010 09:29:17.928000 23462 site-packages/torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::roi_pool
W1010 09:29:17.928000 23462 site-packages/torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::deform_conv2d
/Users/victorpuga/.claude-science/conda/envs/citibike-ml/lib/python3.11/site-packages/torch/onnx/_internal/exporter/_onnx_program.py:486: UserWarning: # The axis name: S will not be used, since it shares the same shape constraints with another axis: S.
  rename_mapping = _dynamic_shapes.create_rename_mapping(


In [17]:
# one call for a whole day, and one call for a single slot: the graph must not depend on the batch
day = data.slots.iloc[test.s0:test.s0 + 96]
args = {'slot_dia': day['slot_dia'].values.astype(np.int64), 'dia_semana': day['dia_semana'].values.astype(np.int64),
        'es_festivo': day['es_festivo'].values.astype(np.float32), 'clima': day[cols].values.astype(np.float32)}
full = sess.run(['lambda'], args)[0]
one = sess.run(['lambda'], {k: v[35:36] for k, v in args.items()})[0]
print('day call', full.shape, '| single slot call', one.shape, '| max abs diff:', float(np.abs(full[35:36] - one).max()))

day call (96, 2515, 4) | single slot call (1, 2515, 4) | max abs diff: 0.0


In [18]:
last = data.Y[:, -28 * 96:].sum((1, 2))
est = data.est
info = {
  'model': onnx_path.name, 'version': 'v3', 'slot_minutes': 15,
  'inputs': {'slot_dia': 'int64 [S], 0..95', 'dia_semana': 'int64 [S], 0 = Monday', 'es_festivo': 'float32 [S], US federal holiday',
             'clima': 'float32 [S, 6]: ' + ', '.join(cols)},
  'outputs': {'lambda': 'float32 [S, n_stations, 4] expected bikes per 15 minutes: salidas_classic, salidas_electric, llegadas_classic, llegadas_electric',
              'alpha': 'float32 [S, n_stations, 4] negative binomial dispersion of one cell, same order'},
  'intervals': {'shocks': plain(shocks), 'alpha_floor': ci.ALPHA_FLOOR,
                'note': 'alpha_idio = max(alpha - (D + H + R + Ds + Hs), alpha_floor); see citibike_intervals.py'},
  'trained_on': {'first_day': str(data.days[0].date()), 'last_day': str(data.days[d_val_end - 1].date())},
  'stations': [dict(station_idx=int(i), station_id=str(r.station_id), name=str(r['name']), lat=float(r.lat), lng=float(r.lng), active=bool(last[i] > 0))
               for i, r in est.iterrows()],
}
(MODELS / 'estaciones_onnx_v3.json').write_text(json.dumps(info, ensure_ascii=False, indent=1))
(MODELS / 'station_ids_v3.json').write_text(json.dumps([s['station_id'] for s in info['stations']]))
torch.save({'state': model_rf.state_dict(), 'cfg': model_rf.cfg, 'mu': feats_rf.mu, 'sd': feats_rf.sd, 'days': (0, d_val_end)}, MODELS / 'demanda_net_v3.pt')
print(f"{len(info['stations']):,} stations, {sum(s['active'] for s in info['stations']):,} active in the last 28 days")
for f in ('demanda_net_v3.pt', 'demanda_net_v3.onnx', 'estaciones_onnx_v3.json', 'station_ids_v3.json'):
  print(f'{f:28s} {(MODELS / f).stat().st_size / 1e6:6.2f} MB')

2,515 stations, 2,423 active in the last 28 days
demanda_net_v3.pt              0.74 MB
demanda_net_v3.onnx            1.24 MB
estaciones_onnx_v3.json        0.44 MB
station_ids_v3.json            0.03 MB


## 8. Summary
- v3 beats v2 and the baseline to beat on every primary metric on the test block, and the gain is largest at station-day scale, where rebalancing decisions are made.
- Cold-start and low-volume stations improve most, because of the neighbourhood prior.
- Station-level intervals are calibrated. City, borough and neighbourhood day totals are too narrow on this block (weather extremes), see section 6.
- A level correction does not help, and there is no persistent drift over five weeks.
- One year of data cannot separate a trend from the season. The model uses weather, not the day of the year, and its forecast for colder days is still about 8% low.